# HMDA Fairness Audit Replication

This notebook reloads the checked-in HMDA fixture, trains the same service model, and reproduces Statistical Parity Difference, Equalized Odds, and Disparate Impact Ratio.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'backend'))

from app.data import load_hmda_frame
from app.modeling import CreditRiskModel

frame = load_hmda_frame(PROJECT / 'backend' / 'data' / 'hmda_il_warren_2023_originated_denied.csv')
frame.head()

In [ ]:
model = CreditRiskModel()
model.fit(frame)
model.runtime.training_summary

In [ ]:
audit = model.audit()['fairness']
audit['summary']

In [ ]:
import pandas as pd

slices = pd.DataFrame(audit['slices'])
slices[['dimension', 'group', 'n', 'approval_rate', 'statistical_parity_difference', 'disparate_impact_ratio', 'four_fifths_rule']]

In [ ]:
sample = model.runtime.sample_applicant
prediction = model.predict(sample)
{k: prediction[k] for k in ['decision', 'probability', 'threshold', 'latency_ms', 'explanation_method']}